In [7]:
from __future__ import print_function, division
import scipy

import keras
import tensorflow as tf
#import tensorflow-addons as tfa
#import tensorflow_addons as tfa
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import Input, Dense, Reshape, Flatten, Dropout, Concatenate
from tensorflow.keras.layers import BatchNormalization, Activation, ZeroPadding2D
#from keras.layers.advanced_activations import LeakyReLU
from tensorflow.keras.layers import LeakyReLU,ReLU,Add, PReLU,add
from tensorflow.keras.layers import UpSampling2D, Conv2D, MaxPooling2D, Conv2DTranspose, SeparableConv2D
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.optimizers import schedules
from tensorflow.keras.layers import MaxPool2D,multiply,Lambda
from keras import backend as K


# Creation of the neural network model



In [8]:
def landslide_attention(img_shape_in,img_shape_dem,channels_out,gf):
    def conv_block(inputs, filters, pool=True):
        x = Conv2D(filters, 3, padding="same")(inputs)

        x = BatchNormalization()(x)
        x = Activation("relu")(x)

        x = Conv2D(filters, 3, padding="same")(x)
        x = BatchNormalization()(x)
        x = Activation("relu")(x)
        if pool:
            p = MaxPool2D((2, 2))(x)
            return x, p
        else:
            return x

    def attention_block_2d(x, g, inter_channel):
        # theta_x(?,g_height,g_width,inter_channel)
        theta_x = Conv2D(inter_channel, [1, 1], strides=[1, 1])(x)
        # phi_g(?,g_height,g_width,inter_channel)
        phi_g = Conv2D(inter_channel, [1, 1], strides=[1, 1])(g)
        # f(?,g_height,g_width,inter_channel)
        f = Activation('relu')(add([theta_x, phi_g]))
        # psi_f(?,g_height,g_width,1)
        psi_f = Conv2D(1, [1, 1], strides=[1, 1])(f)
        rate = Activation('sigmoid')(psi_f)
        att_x = multiply([x, rate])
        return att_x

    def attention_up_and_concate(down_layer, layer):
        in_channel = down_layer.get_shape().as_list()[3]
        up = UpSampling2D(size=(2, 2))(down_layer)
        layer = attention_block_2d(x=layer, g=up, inter_channel=in_channel // 4)
        my_concat = Lambda(lambda x: K.concatenate([x[0], x[1]], axis=3))
        concate = my_concat([up, layer])
        return concate

    inputs_ima = Input(img_shape_in,name="input_image")
    """ Encoder image """
    x1i, p1i = conv_block(inputs_ima, gf)
    x2i, p2i = conv_block(p1i, gf * 2)
    x3i, p3i = conv_block(p2i, gf * 3)
    x4i, p4i = conv_block(p3i, gf * 4)
        
    inputs_dem = Input(img_shape_dem,name="input_dem")
    """ Encoder dem """
    x1d, p1d = conv_block(inputs_dem, gf/2)
    x2d, p2d = conv_block(p1d, gf )
    x3d, p3d = conv_block(p2d, gf * 2)
    x4d, p4d = conv_block(p3d, gf * 3)

    bridge = Concatenate(axis=-1,name="fusion_encoders")([p4i, p4d])
#    bridge = p4d
    b1 = conv_block(bridge, gf * 8,pool=False)

    """ Decoder """
    x = attention_up_and_concate(b1,Concatenate(axis=-1)([x4i, x4d]))
    x = conv_block(x, 4 * gf, pool=False)
    x = attention_up_and_concate(x, Concatenate(axis=-1)([x3i, x3d]))
    x = conv_block(x, 3 * gf, pool=False)
    x = attention_up_and_concate(x,Concatenate(axis=-1)([x2i, x2d]))
    x = conv_block(x, 2 * gf, pool=False)
    x = attention_up_and_concate(x,Concatenate(axis=-1)([x1i, x1d]))
    x = conv_block(x, 2 * gf, pool=False)


    """ Output layer """
    output = Conv2D(channels_out, 1, padding="same", activation="sigmoid")(x)
    return Model([inputs_ima,inputs_dem], output)


In [9]:
img_shape_in=(256,256,3)
img_shape_dem=(256,256,1)
img_shape_out=(256,256,1)
gf=32
channels_out=1


In [10]:
model=landslide_attention(img_shape_in,img_shape_dem,channels_out,gf)
model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_image (InputLayer)       [(None, 256, 256, 3  0           []                               
                                )]                                                                
                                                                                                  
 input_dem (InputLayer)         [(None, 256, 256, 1  0           []                               
                                )]                                                                
                                                                                                  
 conv2d_39 (Conv2D)             (None, 256, 256, 32  896         ['input_image[0][0]']            
                                )                                                           

                                                                                                  
 activation_45 (Activation)     (None, 128, 128, 32  0           ['batch_normalization_37[0][0]'] 
                                )                                                                 
                                                                                                  
 max_pooling2d_9 (MaxPooling2D)  (None, 64, 64, 64)  0           ['activation_37[0][0]']          
                                                                                                  
 max_pooling2d_13 (MaxPooling2D  (None, 64, 64, 32)  0           ['activation_45[0][0]']          
 )                                                                                                
                                                                                                  
 conv2d_43 (Conv2D)             (None, 64, 64, 96)   55392       ['max_pooling2d_9[0][0]']        
          

 ormalization)                                                                                    
                                                                                                  
 activation_50 (Activation)     (None, 16, 16, 256)  0           ['batch_normalization_42[0][0]'] 
                                                                                                  
 conv2d_56 (Conv2D)             (None, 16, 16, 256)  590080      ['activation_50[0][0]']          
                                                                                                  
 batch_normalization_43 (BatchN  (None, 16, 16, 256)  1024       ['conv2d_56[0][0]']              
 ormalization)                                                                                    
                                                                                                  
 activation_51 (Activation)     (None, 16, 16, 256)  0           ['batch_normalization_43[0][0]'] 
          

 ormalization)                                                                                    
                                                                                                  
 activation_59 (Activation)     (None, 64, 64, 96)   0           ['batch_normalization_47[0][0]'] 
                                                                                                  
 up_sampling2d_6 (UpSampling2D)  (None, 128, 128, 96  0          ['activation_59[0][0]']          
                                )                                                                 
                                                                                                  
 concatenate_6 (Concatenate)    (None, 128, 128, 96  0           ['activation_37[0][0]',          
                                )                                 'activation_45[0][0]']          
                                                                                                  
 conv2d_67

                                                                                                  
 activation_66 (Activation)     (None, 256, 256, 64  0           ['batch_normalization_50[0][0]'] 
                                )                                                                 
                                                                                                  
 conv2d_76 (Conv2D)             (None, 256, 256, 64  36928       ['activation_66[0][0]']          
                                )                                                                 
                                                                                                  
 batch_normalization_51 (BatchN  (None, 256, 256, 64  256        ['conv2d_76[0][0]']              
 ormalization)                  )                                                                 
                                                                                                  
 activatio

In [ ]:
from keras.utils.vis_utils import plot_model
# fonction plot_model : on donne un modèle, le nom du fichier de sauvegarde, et deux booleens qui disent qu'on regarde les tailles et les noms des layers
plot_model(model, to_file='fusion.png', show_shapes=True, show_layer_names=True)
